# Text PDF 解析（纯文本类）

**适用**：有完整文本层、无大图的 PDF。含单栏、双栏、学术论文、技术报告等。

**方法**：PyMuPDF 提块 → 检测单/双栏 → 按正确阅读顺序排 → 用字号区分标题/正文/脚注。

**局限**：
- 表格只能提出文字，行列关系丢失
- 公式当普通文本处理，上下标可能乱
- 极端版面（三栏、复杂嵌套）可能顺序错

**依赖**：`pip install pymupdf`

In [2]:
from dataclasses import dataclass
from pathlib import Path
import re

import fitz  # PyMuPDF


@dataclass
class ParsedChunk:
    content: str
    block_type: str  # title / paragraph / footnote / footer
    page: int
    column: str      # left / right / full（占满整行的块）


def _max_font_size(block: dict) -> float:
    """块内最大字号。"""
    sizes = [
        span["size"]
        for line in block.get("lines", [])
        for span in line.get("spans", [])
        if span.get("text", "").strip()
    ]
    return max(sizes) if sizes else 0


def _block_text(block: dict) -> str:
    """提出块内文字，合并连字符断行（如 detec-\n  tion → detection）。"""
    raw = " ".join(
        span["text"]
        for line in block.get("lines", [])
        for span in line.get("spans", [])
    )
    # 合并 word-\n  break 式连字符：去掉「- 」
    raw = re.sub(r"-\s+", "", raw)
    # 压缩多余空白
    return re.sub(r"\s+", " ", raw).strip()


def _detect_columns(blocks: list[dict], page_width: float) -> int:
    """
    简单判断双栏：如果大多数块的右边界不超过页面中线，说明存在双栏布局。
    返回 1 或 2。
    """
    mid = page_width / 2
    # 排除横跨全页的块（标题、页眉等）
    narrow_blocks = [b for b in blocks if (b["bbox"][2] - b["bbox"][0]) < page_width * 0.6]
    if not narrow_blocks:
        return 1
    left_count = sum(1 for b in narrow_blocks if b["bbox"][2] <= mid + 20)
    right_count = sum(1 for b in narrow_blocks if b["bbox"][0] >= mid - 20)
    # 左右都有相当数量的块 → 双栏
    if left_count >= 2 and right_count >= 2:
        return 2
    return 1


def _sort_blocks_by_reading_order(blocks: list[dict], page_width: float, n_columns: int) -> list[tuple[dict, str]]:
    """
    按阅读顺序排块，返回 (block, column_label) 列表。

    单栏：直接按 y 排。
    双栏：横跨全页的块（标题/页眉）按 y 插入；其余先读左栏、再读右栏。
    """
    if n_columns == 1:
        return [(b, "full") for b in sorted(blocks, key=lambda b: b["bbox"][1])]

    mid = page_width / 2
    full, left, right = [], [], []

    for b in blocks:
        x0, _, x1, _ = b["bbox"]
        width = x1 - x0
        if width >= page_width * 0.6:
            full.append(b)
        elif x1 <= mid + 20:
            left.append(b)
        else:
            right.append(b)

    full.sort(key=lambda b: b["bbox"][1])
    left.sort(key=lambda b: b["bbox"][1])
    right.sort(key=lambda b: b["bbox"][1])

    # 把 full 块按 y 插入到左/右栏之间
    result: list[tuple[dict, str]] = []
    left_idx = right_idx = full_idx = 0

    def _next_y(lst, idx):
        return lst[idx]["bbox"][1] if idx < len(lst) else float("inf")

    # 先把 full 块中在左栏内容之前的插入
    while full_idx < len(full) and _next_y(full, full_idx) < _next_y(left, 0):
        result.append((full[full_idx], "full"))
        full_idx += 1

    # 左栏全部读完
    for b in left:
        result.append((b, "left"))

    # 左栏和右栏之间的 full 块
    while full_idx < len(full) and _next_y(full, full_idx) < _next_y(right, 0):
        result.append((full[full_idx], "full"))
        full_idx += 1

    # 右栏全部读完
    for b in right:
        result.append((b, "right"))

    # 剩余 full 块
    for b in full[full_idx:]:
        result.append((b, "full"))

    return result


def parse_text_pdf(pdf_path: str | Path) -> list[ParsedChunk]:
    """
    解析 text 类型 PDF，支持单栏和双栏。

    分类依据：
    - 字号 >= 正文均值 1.4 倍 → title
    - 字号 <= 正文均值 0.85 倍 且在页面下半段 → footnote
    - 位置在页面底部 5% → footer
    - 其余 → paragraph
    """
    doc = fitz.open(Path(pdf_path))
    chunks: list[ParsedChunk] = []

    for page_index, page in enumerate(doc, start=1):
        page_height = page.rect.height
        page_width = page.rect.width

        raw_blocks = [b for b in page.get_text("dict")["blocks"] if b["type"] == 0]
        if not raw_blocks:
            continue

        # 平均字号（正文基准）
        all_sizes = [
            span["size"]
            for b in raw_blocks
            for line in b.get("lines", [])
            for span in line.get("spans", [])
            if span.get("text", "").strip()
        ]
        avg_size = sum(all_sizes) / len(all_sizes) if all_sizes else 11

        n_cols = _detect_columns(raw_blocks, page_width)
        ordered = _sort_blocks_by_reading_order(raw_blocks, page_width, n_cols)

        footer_parts: list[str] = []

        for block, col_label in ordered:
            text = _block_text(block)
            if not text:
                continue

            y_center = (block["bbox"][1] + block["bbox"][3]) / 2
            font_size = _max_font_size(block)

            if y_center > page_height * 0.95:
                footer_parts.append(text)
            elif font_size >= avg_size * 1.4:
                chunks.append(ParsedChunk(text, "title", page_index, col_label))
            elif font_size <= avg_size * 0.85 and y_center > page_height * 0.5:
                # 小字 + 下半页 → 脚注
                chunks.append(ParsedChunk(text, "footnote", page_index, col_label))
            else:
                chunks.append(ParsedChunk(text, "paragraph", page_index, col_label))

        if footer_parts:
            chunks.append(ParsedChunk(" ".join(footer_parts), "footer", page_index, "full"))

    doc.close()
    return chunks

## 测试：双栏学术论文

`multi-column.pdf` — 13 页双栏论文，含表格（Table 1–7）和公式。

重点观察：
- 阅读顺序是否正确（左栏先、右栏后）
- 标题/章节号是否识别
- 表格和公式会原样输出文字，结构不保留

In [ ]:
from pathlib import Path

chunks = parse_text_pdf(Path("workshops/其他格式/pdf/multi-column.pdf"))

print(f"共 {len(chunks)} 块\n")

# 只看第 1 页，方便核对阅读顺序
p1 = [c for c in chunks if c.page == 1]
print(f"第1页: {len(p1)} 块")
print("=" * 60)
for i, c in enumerate(p1, start=1):
    preview = c.content[:120] + "..." if len(c.content) > 120 else c.content
    print(f"[{i}] {c.block_type} ({c.column})")
    print(f"    {preview}\n")

共 302 块

第1页: 14 块
[1] paragraph (full)
    Dense Passage Retrieval for Open-Domain Question Answering

[2] paragraph (full)
    Ledell Wu, Sergey Edunov, Danqi Chen ‡ , Wen-tau Yih Facebook AI † University of Washington ‡ Princeton University { vla...

[3] paragraph (left)
    Abstract

[4] paragraph (left)
    Open-domain question answering relies on efﬁcient passage retrieval to select candidate contexts, where traditional spar...

[5] title (left)
    arXiv:2004.04906v3 [cs.CL] 30 Sep 2020

[6] paragraph (left)
    1 Introduction

[7] paragraph (left)
    Open-domain question answering (QA) ( Voorhees , 1999 ) is a task that answers factoid questions using a large collectio...

[8] paragraph (left)
    ∗ Equal contribution 1 The code and trained models have been released at https://github.com/facebookresearch/DPR .

[9] footnote (left)
    2 For instance, the exact match score on SQuAD v1.1 drops from above 80% to less than 40% ( Yang et al. , 2019a ).

[10] paragraph (right)
    V

In [ ]:
# 整体统计
from collections import Counter

type_counts = Counter(c.block_type for c in chunks)
print("块类型分布:", dict(type_counts))

titles = [c for c in chunks if c.block_type == "title"]
print(f"\n识别到的标题（共 {len(titles)} 个）:")
for c in titles:
    print(f"  第{c.page}页 | {c.content[:80]}")